# Bias mitigiation using the synthesized data
---
The goal of this notebook is to use different tools to check whether the automatic bias detected is also automatically mitigatable. Intermediate results are displayed directly here, for summarized data, have a look at the thesis.

In [ ]:
import sys

sys.path.append("../src")  # go to parent dir
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from data_utils import split_data, prepare_data_fair_learning
from models_utils import evaluate_model, evaluate_model, train_and_evaluate_pipeline, calculate_utility_metrics
from fairness_utils import (
    search_bias,
    evaluate_fairness_score,
    explain_bias,
    encode_protected_attributes,
    evaluate_fairness,
    convert_to_standardDataset,
    reweight_mitigation,
    get_fair_learning_scoring,
    train_and_evaluate_fairness_pipeline
)
from sklearn import clone
from aif360.algorithms.preprocessing import Reweighing, DisparateImpactRemover, LFR
from sklearn.model_selection import GridSearchCV
from aif360.sklearn.preprocessing import LearnedFairRepresentations

warnings.filterwarnings("ignore")
random_state = 12041500

## Load data
We start by loading the respective data from the `./data` directory.

In [2]:
def load_data():
    df_train = pd.read_json("../data/synthetic_data_CTGANSynthesizer.json")
    df_test = pd.read_json("../data/testset.json")

    df_train = df_train.drop(columns=["fnlwgt"])
    df_test = df_test.drop(columns=["fnlwgt"])
    return df_train, df_test


ratio_features = ["age", "capital-gain", "capital-loss", "hours-per-week"]
ordinal_features = ["education-num"]
nominal_features = [
    "workclass",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
]
target = "income"

In [3]:
df_train, df_test = load_data()
df_train

,age,workclass,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,income
0,21,Private,9,Never-married,Handlers-cleaners,Husband,Other,Male,0,0,20,0
1,30,Private,6,Married-spouse-absent,Adm-clerical,Not-in-family,White,Female,3,0,40,0
2,47,Private,10,Never-married,Adm-clerical,Not-in-family,Black,Female,0,0,36,0
3,46,Local-gov,14,Divorced,Prof-specialty,Unmarried,White,Female,8,0,40,0
4,63,Private,10,Married-civ-spouse,Farming-fishing,Husband,White,Male,14,0,20,0
...,...,...,...,...,...,...,...,...,...,...,...,...
39068,30,Private,10,Never-married,Other-service,Own-child,Black,Female,4542,0,40,0
39069,32,Private,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,34,0,40,0
39070,27,Private,10,Never-married,Prof-specialty,Other-relative,Other,Male,0,1,40,0
39071,28,Private,6,Never-married,Machine-op-inspct,Other-relative,White,Male,6,0,40,0


## Baseline Classifier
This sections simply trains and evaluates our baseline. We simply use a Logistic Regression because it should be an easy but robust classifier.

In [4]:
clf = LogisticRegression(max_iter=1000, random_state=random_state)

In [11]:
## Train baseline
model, _ = train_and_evaluate_pipeline(
    clf, nominal_features, df_train, df_test, target, drop_na=True, verbose=True
)

Metric          : Value          
Accuracy        : 0.838
Precision       : 0.782
Recall          : 0.746
F1              : 0.761


## Fairness Evaluation
Next, we continue with the evaluation of fairness using multiple approaches.

In [7]:
X_train, y_train = split_data(df_train, target, drop_na=True)
probs = pd.Series(model.predict_proba(X_train)[:, 1])

### Search Bias
We use MDSS to perform a search for privileged classed, concerning the favorable label `>50k`.

In [8]:
privileged_subset, _ = search_bias(X_train, y_train, probs, 1, penalty=1)
print(privileged_subset)

({'capital-gain': [0, 1, 4, 5, 6]}, 226.8525)


In [ ]:
_ = evaluate_fairness_score(df_train, privileged_subset[0].keys(), target, verbose=True)

Sensitive Attributes: ['capital-gain']

                         Group Distance  Proportion  Counts   P-Value
capital-gain [37.00, 16383.00]    0.434    0.095449    3563  0.00e+00
    capital-gain [-0.00, 5.00]   -0.137    0.517078   19302 1.48e-323
   capital-gain [23.00, 37.00]    0.253    0.097458    3638 3.41e-256
   capital-gain [15.00, 23.00]    0.087    0.106352    3970  1.82e-38
    capital-gain [5.00, 10.00]   -0.047    0.097645    3645  2.44e-13

Weighted Mean Statistical Distance: 0.1648489023109496


We inspect the subset size, expected probability, and classifier’s probability for protected attributes and privileged classes. We do this to get an better feel for the "problem size" we are facing.

In [21]:
explain_bias(df_train, probs, target, privileged_subset[0])

Our detected privileged group has a size of 19340, we observe 0.0639 as the average probability of earning >50k, but our model predicts 0.2058


## Fairness Metrics
We continue by calculating the fairness metrics: `SPD, EOD, AOD, DI` and `Theil index`. For this we use the AIF360 toolbox. We start by encoding the protected attributes as 1 or 0 based if the respective value is within a privileged class. Furthermore, we have to set those attributes as indices of the dataframe to make it work with the framework.

In [9]:
df_train_bias = encode_protected_attributes(df_train, list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)
df_test_bias = encode_protected_attributes(df_test, list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)

1744 Na rows removed!
202 Na rows removed!


Lastly, we can compute the respective metrics

In [23]:
_, _ = train_and_evaluate_fairness_pipeline(
    clf,
    nominal_features,
    df_train_bias,
    target,
    privileged_subset,
    drop_na=True,
    verbose=True,
)

Metric                         : Value          
statistical_parity_difference   0.221
average_odds_difference         0.184
equal_opportunity_difference    0.302
disparate_impact                5.935
theil_index                     0.113


## Mitigation
Next, we use different mitigation techniques to solve the problem. More information can be found within the thesis itself.

### Reweighting
We start by creating the privileged and unprivileged groups.

In [24]:
# create (un)privileged groups
privileged_groups = [{key: 1 for key in list(privileged_subset[0].keys())}]
unprivileged_groups = [{key: 0 for key in list(privileged_subset[0].keys())}]

Next, we convert our datasets to StandardDatasets implemented through AIF360, such that their provided implementations work with our data.

In [ ]:
# convert standard dataset (sd)
sd_train = convert_to_standardDataset(df_train.dropna(), nominal_features, target, 1, list(privileged_subset[0].keys()), list(privileged_subset[0].values()))
sd_test = convert_to_standardDataset(df_test.dropna(), nominal_features, target, 1, list(privileged_subset[0].keys()), list(privileged_subset[0].values()))

Lastly, we fit and transform the dataset using the Reweighing strategy.

In [ ]:
RW = Reweighing(unprivileged_groups=unprivileged_groups, privileged_groups=privileged_groups)
sd_reweigh = RW.fit_transform(sd_train)

#### Baseline
We again establish a baseline, such that comparing results is easier

In [27]:
model = clone(clf)
model.fit(sd_train.features, sd_train.labels.ravel())

_ = evaluate_model(model, sd_test.features, sd_test.labels.ravel(), verbose=True)

Metric          : Value          
Accuracy        : 0.826
Precision       : 0.801
Recall          : 0.680
F1              : 0.710


In [28]:
_ = evaluate_fairness(df_train_bias[target], model.predict(sd_train.features), list(privileged_subset[0].keys()), verbose=True)

Metric                         : Value          
statistical_parity_difference   0.294
average_odds_difference         0.326
equal_opportunity_difference    0.533
disparate_impact                19.095
theil_index                     0.108


#### Mitigated Model
Next, we apply the transform dataset, to see whether it has an positive effect.

In [29]:
model = clone(clf)
model.fit(sd_reweigh.features, sd_reweigh.labels.ravel(), sample_weight=sd_reweigh.instance_weights)

_ = evaluate_model(model, sd_test.features, sd_test.labels.ravel(), verbose=True)

Metric          : Value          
Accuracy        : 0.815
Precision       : 0.747
Recall          : 0.754
F1              : 0.750


In [30]:
_ = evaluate_fairness(df_train_bias[target], model.predict(sd_train.features), list(privileged_subset[0].keys()), verbose=True)

Metric                         : Value          
statistical_parity_difference   0.107
average_odds_difference         -0.002
equal_opportunity_difference    0.002
disparate_impact                2.234
theil_index                     0.134


#### Bias detection & mitigation until bias free
Lastly, we want to experiment, whether we could apply bias detection until a classifier is bias free. We would like to observe the behaviour of metrics over iterations.

In [39]:
df_fairness_metrics, df_metrics = pd.DataFrame(
    columns=[
        "statistical_parity_difference",
        "average_abs_odds_difference",
        "equal_opportunity_difference",
        "disparate_impact",
        "theil_index",
    ]
), pd.DataFrame(
    columns=['acc', 'prec', 'rec', 'f1']
)

In [ ]:
df_train, df_test = load_data()

weights, weights_hist, max_iter = None, [None], 25
for i in tqdm(range(max_iter)):
    weights, model_metrics, fair_metrics = reweight_mitigation(
        clf,
        nominal_features,
        target,
        df_train,
        df_test,
        penalty=1,
        sample_weights=weights,
    )
    if model_metrics is None and fair_metrics is None and weights is None:
        break

    df_metrics.loc[f"mitigation_{i}"] = model_metrics.values()
    df_fairness_metrics.loc[f"mitigation_{i}"] = fair_metrics.values()
    weights_hist.append(weights)

In [ ]:
df_fairness_metrics.to_json(path_or_buf="../results/fairness_metrics_25_iterations_synthetic.json")
df_metrics.to_json(path_or_buf="../results/utility_metrics_25_iterations_synthetic.json")

### Fair Learning
Next, we consider Fair Learning, for this we reload the data and prepare it for the fair learning technique.

In [9]:
df_train, df_test = load_data()

In [18]:
df_train_bias = encode_protected_attributes(df_train.dropna(), list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)
df_test_bias = encode_protected_attributes(df_test.dropna(), list(privileged_subset[0].keys()), list(privileged_subset[0].values()), verbose=True)

X_train, y_train, X_test, y_test = prepare_data_fair_learning(df_train_bias, df_test_bias, nominal_features, target)

In [11]:
X_train['capital-gain'] = X_train.index
X_test['capital-gain'] = X_test.index

In [12]:
max_delta = get_fair_learning_scoring(list(privileged_subset[0].keys()))

Next, we prepare the algorithm and the params.

In [13]:
lfr = LearnedFairRepresentations(
    list(privileged_subset[0].keys()),
    n_prototypes=10,
    max_iter=10,
    random_state=random_state,
)

In [ ]:
params = {
    "reconstruct_weight": [1e-2, 1e-3, 1e-4],
    "target_weight": [100, 1000],
    "fairness_weight": [0, 100, 1000],
}

Lastly, we perform grid search to get to the best results.

In [ ]:
grid = GridSearchCV(lfr, params, scoring=max_delta, cv=3, n_jobs=-1).fit(
    X_train, y_train, priv_group=(1,) * len(list(privileged_subset[0].keys()))
)
res = pd.DataFrame(grid.cv_results_)

#### Baseline
Again, we start by implementing the baseline.

In [ ]:
model = clone(clf)
model.fit(X_train, y_train)

_ = evaluate_model(model, X_test, y_test, verbose=True)

Metric         Value               
Accuracy       0.827
Precision      0.802
Recall         0.682
F1             0.711


In [ ]:
_ = evaluate_fairness(df_train_bias[target], model.predict(X_train), list(privileged_subset[0].keys()), verbose=True)

Metric                          Value               
statistical_parity_difference   0.293
average_odds_difference         0.325
equal_opportunity_difference    0.531
disparate_impact                18.818
theil_index                     0.108


#### Using Grid itself
Next, we simply use the trained grid-search object to perform the predictions.

In [ ]:
_ = evaluate_model(grid, X_test, y_test, verbose=True)

Metric         Value               
Accuracy       0.759
Precision      0.627
Recall         0.522
F1             0.488


In [ ]:
_ = evaluate_fairness(df_train_bias[target], grid.predict(X_train), list(privileged_subset[0].keys()), verbose=True)

Metric                          Value               
statistical_parity_difference   0.019
average_odds_difference         0.019
equal_opportunity_difference    0.026
disparate_impact                10.277
theil_index                     0.223


#### Transforming data
Next, we use the grid to transform the training data into the respective object and use its results.

In [ ]:
model = clone(clf)
model.fit(grid.transform(X_train), y_train)

_ = evaluate_model(model, X_test, y_test, verbose=True)

Metric         Value               
Accuracy       0.450
Precision      0.554
Recall         0.564
F1             0.447


In [ ]:
_ = evaluate_fairness(df_train_bias[target], model.predict(X_test), list(privileged_subset[0].keys()), verbose=True)

Metric                          Value               
statistical_parity_difference   0.084
average_odds_difference         0.064
equal_opportunity_difference    0.054
disparate_impact                1.116
theil_index                     0.085


##### Also Transforming test data
Lastly, we use the grid to also transform the test data into the respective object and use its results.

In [ ]:
_ = evaluate_model(model, grid.transform(X_test), y_test, verbose=True)

Metric         Value               
Accuracy       0.766
Precision      0.662
Recall         0.587
F1             0.596


In [ ]:
_ = evaluate_fairness(df_train_bias[target], model.predict(grid.transform(X_train)), list(privileged_subset[0].keys()), verbose=True)

Metric                          Value               
statistical_parity_difference   0.123
average_odds_difference         0.100
equal_opportunity_difference    0.141
disparate_impact                3.928
theil_index                     0.177


Multiple Runs cannot be done, because of the dataset already needs to be one hot encoded for the mitigation, making the `search_bias` function unnecessary and returning not viable resolutions.

### Fair Learning (AIF360)
Next, we look at the Fair Learning implementation by AIF360.

In [15]:
df_train, df_test = load_data()
ds_train = convert_to_standardDataset(df_train.dropna(), nominal_features, target, 1, list(privileged_subset[0].keys()), list(privileged_subset[0].values()))
ds_test = convert_to_standardDataset(df_test.dropna(), nominal_features, target, 1, list(privileged_subset[0].keys()), list(privileged_subset[0].values()))

In [51]:
privileged_groups = [{key: 1 for key in list(privileged_subset[0].keys())}]
unprivileged_groups = [{key: 0 for key in list(privileged_subset[0].keys())}]

In [52]:
TR = LFR(unprivileged_groups=unprivileged_groups,
         privileged_groups=privileged_groups,
         k=10, Ax=0.01, Ay=1000, Az=0,
         verbose=1,
         seed=random_state
)

TR = TR.fit(ds_train, maxiter=5000, maxfun=1000)

step: 0, loss: 605.8719525156314, L_x: 4089.307287970777,  L_y: 0.5649788796359236,  L_z: 0.000442105866442874
step: 250, loss: 605.8719527432613, L_x: 4089.307287989528,  L_y: 0.564978879863366,  L_z: 0.000442105806862049
step: 500, loss: 605.8719475746043, L_x: 4089.307287967263,  L_y: 0.5649788746949316,  L_z: 0.0004421057161778499
RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =          500     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  6.05872D+02    |proj g|=  2.78235D+01
step: 750, loss: 559.2910323520119, L_x: 4088.5294761775235,  L_y: 0.5184057375902368,  L_z: 0.0004318659186486676
step: 1000, loss: 559.291034705393, L_x: 4088.5294761868945,  L_y: 0.518405739943524,  L_z: 0.00043186608255237833

At iterate    1    f=  5.59291D+02    |proj g|=  1.09570D+01

           * * *

Tit   = total number of iterations
Tnf   = total number of function evaluations
Tnint = total number of segments explored d

Transform training data and align features

In [67]:
ds_train_lfr = TR.transform(ds_train)

#### Baseline
We again start by establishing a baseline.

In [56]:
model = clone(clf)
model.fit(ds_train.features, ds_train.labels.ravel())

_ = evaluate_model(model, ds_test.features, ds_test.labels.ravel(), verbose=True)

Metric          : Value          
Accuracy        : 0.826
Precision       : 0.801
Recall          : 0.680
F1              : 0.710


In [57]:
_ = evaluate_fairness(df_train_bias[target], model.predict(ds_train.features), list(privileged_subset[0].keys()), verbose=True)

Metric                         : Value          
statistical_parity_difference   0.294
average_odds_difference         0.326
equal_opportunity_difference    0.533
disparate_impact                19.095
theil_index                     0.108


#### Mitigation
Since the tool does simply convert all labels to 0, its easy to see why those results are not promising and should not be further considered.

In [68]:
model = clone(clf)
model.fit(ds_train_lfr.features, ds_train_lfr.labels.ravel())

_ = evaluate_model(model, ds_test.features, ds_test.labels.ravel(), verbose=True)

ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: 0.0

In [16]:
_ = calculate_utility_metrics(ds_test.labels.ravel(), [0] * len(ds_test.labels), verbose=True)

Metric          : Value          
Accuracy        : 0.759
Precision       : 0.380
Recall          : 0.500
F1              : 0.432


In [64]:
_ = evaluate_fairness(df_train_bias[target], np.array([0]*len(df_train_bias)), list(privileged_subset[0].keys()), verbose=True)

Metric                         : Value          
statistical_parity_difference   0.000
average_odds_difference         0.000
equal_opportunity_difference    0.000
disparate_impact                0.000
theil_index                     0.229


Multiple Runs cannot be done, because of using the StandardDataset by AIF360, which uses one-way one-hot encoding.

### Disparate Impact Remover
Lastly, we apply the DI-Remover to improve DI. For this, we start by reloading and converting the datasets.

In [46]:
df_train, df_test = load_data()
df_train = convert_to_standardDataset(df_train.dropna(), nominal_features, target, 1, list(privileged_subset[0].keys()), list(privileged_subset[0].values()))
df_test = convert_to_standardDataset(df_test.dropna(), nominal_features, target, 1, list(privileged_subset[0].keys()), list(privileged_subset[0].values()))

Next, we apply the DI-Remover with different levels of repairment.

In [42]:
fairness_metrics = []
utility_metrics = []
for level in tqdm(np.linspace(0, 1, 10)):
    di = DisparateImpactRemover(repair_level=level)
    ds_train = di.fit_transform(df_train)
    ds_test = di.fit_transform(df_test)
    
    X_train, y_train = ds_train.features, ds_train.labels.ravel()
    X_test, y_test = ds_test.features, ds_test.labels.ravel()
    
    model = clone(clf)
    model.fit(X_train, y_train)

    utility_metrics.append(evaluate_model(model, X_test, y_test, verbose=False))
    fairness_metrics.append(evaluate_fairness(df_train_bias[target], model.predict(X_train), list(privileged_subset[0].keys()), verbose=False))


100%|██████████| 10/10 [01:46<00:00, 10.64s/it]


Lastly, we select the best performing result, based on DI and output fairness and utility metrics.

In [38]:
max_index, max_disparate_impact_row = max(enumerate(fairness_metrics), key=lambda x: x[1]['disparate_impact'])
max_disparate_impact_row

{'statistical_parity_difference': 0.2940092562267429,
 'average_odds_difference': 0.3263848052479457,
 'equal_opportunity_difference': 0.5325510761059932,
 'disparate_impact': 19.095289689901932,
 'theil_index': 0.1077590833288912}

In [39]:
utility_metrics[max_index]

{'Accuracy': 0.8263823560154698,
 'Precision': 0.8009893116559623,
 'Recall': 0.680496775358145,
 'F1': 0.7095284659429854}

Multiple Runs cannot be done, because of using the StandardDataset by AIF360, which uses one-way one-hot encoding.